In [2]:
import requests
response = requests.get("https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Cell_Phones_and_Accessories.jsonl.gz")

if response.status_code == 200:
    with open('meta_Cell_Phones_and_Accessories.jsonl.gz', 'wb') as f:
        f.write(response.content)

In [3]:
import pandas as pd

df = pd.read_json('meta_Cell_Phones_and_Accessories.jsonl.gz', lines=True, chunksize=10000)
df = next(df)
df2 = df.to_csv('dataset.csv', index=False)

In [4]:
df = pd.read_csv('dataset.csv')
df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Cell Phones & Accessories,ARAREE Slim Diary Cell Phone Case for Samsung ...,3.8,5,['Genuine Cow leather with 6 different colors'...,"[""JUST LOOK, You can tell the difference. Make...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],araree,"['Cell Phones & Accessories', 'Cases, Holsters...",{'Product Dimensions': '3.35 x 0.59 x 6.18 inc...,B013SK1JTY,NaN,NaN,NaN
1,Cell Phones & Accessories,Bastmei for OnePlus 7T Case Extremely Light Ul...,4.4,177,['Ultra-thin & Ultra-light: The ultra slim fit...,[],11.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Bastmei,"['Cell Phones & Accessories', 'Cases, Holsters...",{'Package Dimensions': '7.6 x 4.29 x 0.75 inch...,B07ZPSG8P5,NaN,NaN,NaN
2,Cell Phones & Accessories,Wireless Fones Branded New Iphone 5C/LITE Hot ...,4.0,2,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],WIRELESS FONES,"['Cell Phones & Accessories', 'iPhone Accessor...","{'Item model number': 'Apple Iphone 5C', 'Othe...",B00GKR3L12,NaN,NaN,NaN
3,Cell Phones & Accessories,"iPhone 6 Plus + Case, DandyCase Perfect PATTER...",4.0,15,"['Slim-Fit design for the iPhone 6 Plus (5.5"" ...",['Case does not need to be removed for chargin...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],DandyCase,"['Cell Phones & Accessories', 'iPhone Accessor...",{'Product Dimensions': '5.43 x 0.28 x 2.64 inc...,B00PB8U8BW,NaN,NaN,NaN
4,Cell Phones & Accessories,"Case for Galaxy S6/S6 Edge, Thin Translucent V...",4.0,1,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],7Pite,"['Cell Phones & Accessories', 'Cases, Holsters...",{'Package Dimensions': '8.31 x 3.74 x 0.55 inc...,B07D3RHSRV,NaN,NaN,NaN


In [5]:
print("the dataset columns:\n", df.columns.to_list())
print("DF info:\n", df.info)
print("DF stats:\n", df.describe())
print("DF shape: ", df.shape)
print("DF dtypes:\n", df.dtypes)

the dataset columns:
 ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']
DF info:
 <bound method DataFrame.info of                   main_category  \
0     Cell Phones & Accessories   
1     Cell Phones & Accessories   
2     Cell Phones & Accessories   
3     Cell Phones & Accessories   
4     Cell Phones & Accessories   
...                         ...   
9995  Cell Phones & Accessories   
9996  Cell Phones & Accessories   
9997  Cell Phones & Accessories   
9998  Cell Phones & Accessories   
9999  Cell Phones & Accessories   

                                                  title  average_rating  \
0     ARAREE Slim Diary Cell Phone Case for Samsung ...             3.8   
1     Bastmei for OnePlus 7T Case Extremely Light Ul...             4.4   
2     Wireless Fones Branded New Iphone 5C/LITE Hot ...             4.0   
3     i

In [6]:
df["parent_asin"].isnull().sum()

np.int64(0)

In [7]:
import ast

keep_cols = ['parent_asin', 'title', 'store', 'categories',
    'features', 'description', 'price', 'average_rating']

df_clean = df[keep_cols].copy()

def clean_list_column(val):
  if pd.isna(val):
    return ""
  if isinstance(val, str) and val.startswith('['):
    try:
      val = ast.literal_eval(val)
    except:
      return ""
  if isinstance(val, list):
    return " ".join(str(item) for item in val).strip()
  return str(val)


list_cols = ['categories', 'features', "description"]
for col in list_cols:
  df_clean[col] = df_clean[col].apply(clean_list_column)

df_clean["title"] = df_clean["title"].fillna("")
df_clean["store"] = df_clean['store'].fillna("")
df_clean["price"] = df_clean['price'].fillna(-1.0)
df_clean["average_rating"] = df_clean['average_rating'].fillna(0.0)

df_clean["metadata"] = df_clean["store"] + " " + df_clean["categories"] + " " + df_clean["title"] + " " + df_clean["features"] + " " + df_clean["description"]
df_clean["metadata"] = df_clean["metadata"].str.lower()

In [8]:
df_final = df_clean[['parent_asin', 'title', 'price', 'average_rating', 'metadata']]

In [9]:
print(df_final.head())
df_final.info()

  parent_asin                                              title  price  \
0  B013SK1JTY  ARAREE Slim Diary Cell Phone Case for Samsung ...  -1.00   
1  B07ZPSG8P5  Bastmei for OnePlus 7T Case Extremely Light Ul...  11.98   
2  B00GKR3L12  Wireless Fones Branded New Iphone 5C/LITE Hot ...  -1.00   
3  B00PB8U8BW  iPhone 6 Plus + Case, DandyCase Perfect PATTER...  -1.00   
4  B07D3RHSRV  Case for Galaxy S6/S6 Edge, Thin Translucent V...  -1.00   

   average_rating                                           metadata  
0             3.8  araree cell phones & accessories cases, holste...  
1             4.4  bastmei cell phones & accessories cases, holst...  
2             4.0  wireless fones cell phones & accessories iphon...  
3             4.0  dandycase cell phones & accessories iphone acc...  
4             4.0  7pite cell phones & accessories cases, holster...  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column     

In [10]:
df_final.to_csv('cbf_data.csv')